# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Ibrahim-1rfan/Flyrank-/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

Files and Directory Addresses in Dataset


In [2]:
from huggingface_hub import HfFileSystem
from google.colab import userdata

hf_token = userdata.get('HF_TOKEN')
fs = HfFileSystem(token=hf_token)

# List the root directories in the warehouse
directories = fs.ls("datasets/FlyRank/internship-warehouse", detail=False)
print("Available folders/tables in the warehouse:")
for d in directories:
    print(d)

Available folders/tables in the warehouse:
datasets/FlyRank/internship-warehouse/fact_content_daily_performance
datasets/FlyRank/internship-warehouse/.gitattributes
datasets/FlyRank/internship-warehouse/README.md
datasets/FlyRank/internship-warehouse/dim_clients.parquet
datasets/FlyRank/internship-warehouse/dim_content.parquet
datasets/FlyRank/internship-warehouse/fact_content_daily_performance_sample.parquet
datasets/FlyRank/internship-warehouse/fact_content_query_90d.parquet


## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


# 1. Install duckdb quietly
%pip -q install duckdb
import duckdb
from google.colab import userdata

# 2. Pull your secret token from Colab (Do NOT paste it as plain text)
hf_token = userdata.get('HF_TOKEN')

# 3. Connect and pass the token to DuckDB
con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{hf_token}')")

# 4. Point DuckDB to a mid-panel month in that exact repository
# Note: we append the table name and partition (e.g., month=2026-03)
REL = "read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet')"

# 5. Run your first test query
df_test = con.sql(f"SELECT COUNT(*) as row_count FROM {REL}").df()
print(f"Successfully connected! Rows found: {df_test.iloc[0, 0]:,}")

Successfully connected! Rows found: 9,841,378


In [4]:
# 1. Check Fact Table Columns
print("--- Daily Fact Table ---")
con.sql(f"DESCRIBE SELECT * FROM {REL}").show()

# 2. Check Dimension Table Columns
DIM_REL = "read_parquet('hf://datasets/FlyRank/internship-warehouse/dim_content.parquet')"
print("\n--- Content Dimension Table ---")
con.sql(f"DESCRIBE SELECT * FROM {DIM_REL}").show()

--- Daily Fact Table ---
┌────────────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│    column_name     │ column_type │  null   │   key   │ default │  extra  │
│      varchar       │   varchar   │ varchar │ varchar │ varchar │ varchar │
├────────────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ report_date        │ DATE        │ YES     │ NULL    │ NULL    │ NULL    │
│ client_hash_id     │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ content_hash_id    │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ client_has_gsc     │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ client_has_ga4     │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ gsc_data_available │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ ga4_data_available │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ gsc_impressions    │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ gsc_clicks         │ BIGINT      │ YES     │ NULL

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


# Setup the paths to both tables
REL = "read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet')"
DIM_REL = "read_parquet('hf://datasets/FlyRank/internship-warehouse/dim_content.parquet')"

# For the fact table, one row = one page per day.
# If this returns 0, no page has duplicate daily records.
grain_check = con.sql(f"""
    SELECT client_hash_id, content_hash_id, report_date, COUNT(*) as c
    FROM {REL}
    GROUP BY client_hash_id, content_hash_id, report_date
    HAVING c > 1
    LIMIT 5
""").df()
print(f"Grain violations (should be 0): {len(grain_check)}")

# 2. Row Count & Date Span for our selected month
span_check = con.sql(f"""
    SELECT
        COUNT(*) AS total_rows,
        MIN(report_date) AS start_date,
        MAX(report_date) AS end_date
    FROM {REL}
""").df()
print("\nRow Count and Time Window:")
display(span_check)

# 3. Availability Check: How many pages are actually published?
# We use IS TRUE on the dimension table's boolean flag.
availability = con.sql(f"""
    SELECT COUNT(DISTINCT content_hash_id) AS active_pages
    FROM {DIM_REL}
    WHERE is_published IS TRUE
""").df()
print(f"\nActive published pages available: {availability.iloc[0,0]:,}")

In [ ]:
from sklearn.tree import DecisionTreeClassifier

# Build the feature frame and label natively in DuckDB SQL[cite: 3]
df_features = con.sql(f"""
    WITH daily_facts AS (
        SELECT
            client_hash_id,
            content_hash_id,
            SUM(gsc_impressions) AS total_impressions,
            SUM(gsc_clicks) AS total_clicks,
            SUM(gsc_sum_position) AS sum_position,
            MAX(report_date) AS last_date,

            -- THE TRAP components (splitting the month in half)
            SUM(CASE WHEN day(report_date) <= 15 THEN gsc_impressions ELSE 0 END) AS impressions_h1,
            SUM(CASE WHEN day(report_date) > 15 THEN gsc_impressions ELSE 0 END) AS trap_impressions_h2
        FROM {REL}
        WHERE gsc_data_available IS TRUE
        GROUP BY client_hash_id, content_hash_id
    )
    SELECT
        f.client_hash_id,
        f.content_hash_id,

        -- 5 Safe Features
        f.total_impressions AS impressions_window,
        (f.total_clicks * 1.0 / NULLIF(f.total_impressions, 0)) AS ctr,
        (f.sum_position * 1.0 / NULLIF(f.total_impressions, 0)) AS avg_position,
        DATE_DIFF('day', d.content_created_date, f.last_date) AS content_age_days,
        DATE_DIFF('day', d.content_updated_date, f.last_date) AS days_since_last_update,

        -- THE LEAK
        f.trap_impressions_h2,

        -- THE LABEL (Proxy for traffic decay)
        CAST((f.trap_impressions_h2 < f.impressions_h1) AS INTEGER) AS is_declining_label

    FROM daily_facts f
    JOIN {DIM_REL} d ON f.content_hash_id = d.content_hash_id
    WHERE d.is_published IS TRUE
      AND f.total_impressions > 500 -- Filter for mature, visible pages
""").df().fillna(0)

# Define feature sets
features_honest = ["impressions_window", "ctr", "avg_position", "content_age_days", "days_since_last_update"]
features_leaky = features_honest + ["trap_impressions_h2"] # Adding the future data

X_honest = df_features[features_honest]
X_leaky = df_features[features_leaky]
y = df_features["is_declining_label"]

# Train both trees
tree_honest = DecisionTreeClassifier(max_depth=3, random_state=42).fit(X_honest, y)
tree_leaky = DecisionTreeClassifier(max_depth=3, random_state=42).fit(X_leaky, y)

print("--- The Feature Leakage Trap on Real Data ---")
print(f"Score WITH leaked future impressions: {tree_leaky.score(X_leaky, y):.3f} (Too good to be true)")
print(f"Score WITHOUT leaked feature:         {tree_honest.score(X_honest, y):.3f} (The honest baseline)")

# Clean up by removing the leaked column from our working dataframe
df_features = df_features.drop(columns=["trap_impressions_h2"])

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.